In [7]:
import pandas as pd
import geopy.distance
import numpy as np

In [8]:
df = pd.read_csv('../data/raw/all_town_data.csv', index_col=[0])
df = df.replace(to_replace='Ballarat Central', value='Ballarat')

city_centres = pd.read_csv('../data/landing/city_centres.csv', index_col=[0])
city_centres.columns = ['name']
city_centres['city_centre'] = [True for i in range(city_centres.shape[0])]

coastal_towns = pd.read_csv('../data/raw/coastal_towns.csv')
coastal_towns['coastal_towns'] = [True for i in range(coastal_towns.shape[0])]
coastal_towns.columns = ['name', "coastal"]

ballarat_suburbs = pd.read_csv('../data/raw/ballarat_suburbs.csv')
ballarat_suburbs

,name
0,Bakery Hill
1,Ballarat Central
2,Ballarat East
3,Ballarat North
4,Black Hill
5,Bo Peep
6,Bonshaw
7,Brown Hill
8,Bunkers Hill
9,Canadian


In [9]:
# Merge with city centres
df = pd.merge(city_centres, df, on='name', how='outer')
df['city_centre'] = df['city_centre'].fillna(value=False)

In [10]:
# Merge with coastal towns
df = pd.merge(df, coastal_towns, on='name', how='left')
df['coastal'] = df['coastal'].fillna(value=False)
df

,name,city_centre,postalcode,lat,lng,coastal
0,Ballarat,True,3350,-37.56266,143.85038,False
1,Benalla,True,3672,-36.55397,145.97985,False
2,Benalla,True,3671,-36.55113,145.98425,False
3,Benalla,True,3673,-36.55113,145.98425,False
4,Benalla,True,3672,-36.55397,145.97985,False
...,...,...,...,...,...,...
2615,Upper Plenty,False,3756,-37.42079,145.05529,False
2616,Invergordon,False,3636,-36.15014,145.59329,False
2617,Invergordon South,False,3634,-36.24557,145.46864,False
2618,Elaine,False,3334,-37.75879,144.03114,False


In [11]:
# Closest city centre now! 

melb_coords = (-37.81302, 144.96111)

closest_city_centre = []
closest_cc_dist = []
dist_to_melb = []

for _, row in df.iterrows():

    p = geopy.distance.geodesic(melb_coords, (row['lat'], row['lng'])).km

    dist_to_melb.append(p) 

    if row['city_centre']:
        closest_city_centre.append(row['name'])
        closest_cc_dist.append(0)
        continue
    
    q = df[df['city_centre'] == True].copy()
    
    min_dist = np.inf
    closest = None
    for name, qlat, qlong in zip(q['name'], q['lat'], q['lng']):
        curr = geopy.distance.geodesic((qlat, qlong), (row['lat'], row['lng'])).km
        if curr < min_dist:
            min_dist = curr
            closest = name

    closest_city_centre.append(closest)
    closest_cc_dist.append(min_dist)

df['closest_city_centre'] = closest_city_centre
df['closest_cc_dist'] = closest_cc_dist
df['dist_to_melb'] = dist_to_melb

df = df.drop_duplicates().reset_index(drop=True)



In [17]:
gb = df.groupby(['postalcode']).agg(
    lat=('lat', 'mean'),
    long=('lat', 'mean'),
    city_centre=('city_centre', 'first'),
    closest_city_centre=('closest_city_centre', 'first'), 
    coastal=('coastal', 'first'),
    closest_cc_dist=('closest_cc_dist', 'mean'),
    dist_to_melb=('dist_to_melb', 'mean'),
)


In [18]:
gb.to_csv('../data/curated/town_distance_data.csv')